In [1]:
import os
import sys

project_root = "/root/work/tvm-ansor"
os.environ["TVM_HOME"] = f"{project_root}"
os.environ["TVM_LIBRARY_PATH"] = f"{project_root}/build-release"
if f"{project_root}/python" not in sys.path:
    sys.path.insert(0, f"{project_root}/python")

sys.path = [p for p in sys.path if not p.startswith(f"{project_root}/build")]
sys.path.append(f"{project_root}/build-release")
os.environ["LD_LIBRARY_PATH"] = f"{project_root}/build-release:" + os.environ.get("LD_LIBRARY_PATH", "")


import numpy as np
import tvm
from tvm import auto_scheduler
from tvm.auto_scheduler import SketchPolicy

TARGET = tvm.target.Target("cuda")



In [2]:
from modules.common import load_and_register_tasks
tasks = load_and_register_tasks()
concrete_states = {}
sketches_by_idx = {}
for idx, task in enumerate(tasks):
    concrete_state = SketchPolicy(task, params={'sample_init_no_invalid': 1 }, verbose=False).generate_concrete_sketches()
    for i, state in enumerate(concrete_state):
        sketches_by_idx[idx] = (task.desc, f"{task.workload_key}_{i}")
        concrete_states[f"{task.workload_key}_{i}"] = (task, state)

In [16]:
from modules.param_manager import build_symbolic_state
from modules.schedule_generator import ScheduleGenerator
sketch_idx = 178
print(sketches_by_idx[sketch_idx])
sample = list(concrete_states.values())[sketch_idx]
sym_state = build_symbolic_state(sample[0], sample[1])
sg = ScheduleGenerator(sym_state, task=sample[0], base_state=sample[1])
sym_state

('vm_mod_fused_nn_conv2d_add_add_clip_divide_multiply_5', '["8b290ca0f393be6ffa536c91e9bafd22", [1, 15, 15, 80], [1, 1, 80, 184], [1, 1, 1, 184], [1, 15, 15, 184]]_0')


Placeholder: p0, p1, p2
blockIdx.x b.0@i.0@j.0@c.0@ (0,ceil(ceil(ceil(15/(min(sp_8_2*sp_8_3,15)))/(min(sp_8_1,ceil(15/(min(sp_8_2*sp_8_3,15))))))/(min(sp_8_0,ceil(ceil(15/(min(sp_8_2*sp_8_3,15)))/(min(sp_8_1,ceil(15/(min(sp_8_2*sp_8_3,15)))))))))*ceil(ceil(ceil(15/(min(sp_9_2*sp_9_3,15)))/(min(sp_9_1,ceil(15/(min(sp_9_2*sp_9_3,15))))))/(min(sp_9_0,ceil(ceil(15/(min(sp_9_2*sp_9_3,15)))/(min(sp_9_1,ceil(15/(min(sp_9_2*sp_9_3,15)))))))))*ceil(ceil(ceil(672/(min(sp_10_2*sp_10_3,672)))/(min(sp_10_1,ceil(672/(min(sp_10_2*sp_10_3,672))))))/(min(sp_10_0,ceil(ceil(672/(min(sp_10_2*sp_10_3,672)))/(min(sp_10_1,ceil(672/(min(sp_10_2*sp_10_3,672))))))))))
  vthread b.1@i.1@j.1@c.1@ (0,min(sp_8_0,ceil(ceil(15/(min(sp_8_2*sp_8_3,15)))/(min(sp_8_1,ceil(15/(min(sp_8_2*sp_8_3,15)))))))*min(sp_9_0,ceil(ceil(15/(min(sp_9_2*sp_9_3,15)))/(min(sp_9_1,ceil(15/(min(sp_9_2*sp_9_3,15)))))))*min(sp_10_0,ceil(ceil(672/(min(sp_10_2*sp_10_3,672)))/(min(sp_10_1,ceil(672/(min(sp_10_2*sp_10_3,672))))))))
    threadIdx.

In [17]:
initial_domains = {}
for name in sg._all_sp_names:
    step_idx = int(name.split("_")[1])
    extent = sg._sp_extents.get(step_idx)
    if extent is None:
        initial_domains[name] = 1
    else:
        initial_domains[name] = [1, extent]

initial_preview_sym_map = {name: None for name in sg.s.sym_map}
initial_fixed_from_domains = {}
for name, dom in initial_domains.items():
    if isinstance(dom, int):
        initial_preview_sym_map[name] = dom
        initial_fixed_from_domains[name] = dom
    elif isinstance(dom, list) and len(dom) == 2 and dom[0] == dom[1]:
        initial_preview_sym_map[name] = dom[0]
        initial_fixed_from_domains[name] = dom[0]

print("[Fixed From Domains Before Generation]")
print(initial_fixed_from_domains)

print("\n[Constraints Before Generation]")
print(
    sg.get_constraints_with_assignment_str(
        sym_map=initial_preview_sym_map,
        include_vars=True,
        include_eval=True,
    )
)


[Fixed From Domains Before Generation]
{'sp_7_0': 1, 'sp_7_1': 1, 'sp_7_2': 1, 'sp_7_3': 1}

[Constraints Before Generation]
vectorize: sp_41_0 * 4 <= 16
  vars: sp_41_0
  eval: (partial)
vectorize:
  min((((ceil(ceil(ceil(672 / (sp_10_2 * sp_10_3)) / sp_10_1) / sp_10_0)) * sp_10_1 * sp_10_0 - 1) * min(672, sp_10_2 * sp_10_3) + min(min(672, sp_10_2 * sp_10_3), sp_10_3) * min(ceil(min(672, sp_10_2 * sp_10_3) / sp_10_3), sp_10_2)) * min(3, sp_11_1) * min(ceil(3 / sp_11_1), sp_11_0) * min(3, sp_12_1) * min(ceil(3 / sp_12_1), sp_12_0), sp_36_0)
  * 4
  <= 16
  vars: sp_10_0, sp_10_1, sp_10_2, sp_10_3, sp_11_0, sp_11_1, sp_12_0, sp_12_1, sp_36_0
  eval: (partial)
shared_memory:
  ((sp_10_1 * sp_10_0 * (mod(((ceil(ceil(ceil(15 / (sp_8_2 * sp_8_3)) / sp_8_1) / sp_8_0)) * (ceil(ceil(ceil(15 / (sp_9_2 * sp_9_3)) / sp_9_1) / sp_9_0)) * (ceil(ceil(ceil(672 / (sp_10_2 * sp_10_3)) / sp_10_1) / sp_10_0)) - 1), (ceil(ceil(ceil(672 / (sp_10_2 * sp_10_3)) / sp_10_1) / sp_10_0))) + 1) - 1) * min(672, sp

In [18]:
import random

phase_entries = sg.get_var_order_phase_entries()

print("[Phase Order]")
for entry in phase_entries:
    shown_vars = entry["vars"] if entry["vars"] else ["<empty>"]
    print(entry["name"], shown_vars)

original_sym_map = dict(sg.s.sym_map)

for entry in phase_entries:
    print(f"\n=== {entry['name']} ===")
    prefix_state = sg.param_sampler.randomize_params_prefix(
        entry["name"],
        rng=random.Random(0),
        max_retries=1,
    )
    sg.s.sym_map.update(original_sym_map)

    preview_sym_map = {name: None for name in original_sym_map}
    preview_sym_map.update(prefix_state["fixed_values"])
    preview_sym_map.update(prefix_state["params"])

    print("[Phase Vars]")
    print(entry["vars"] if entry["vars"] else ["<empty>"])

    print("\n[Params At This Phase]")
    print(prefix_state["params"])

    print("\n[Fixed From Domains]")
    fixed_from_domains = {
        name: value
        for name, value in prefix_state["fixed_values"].items()
        if name not in prefix_state["params"]
    }
    print(fixed_from_domains)

    print("\n[Domains After Phase]")
    print(prefix_state["domains"])

    print("\n[Constraints After Phase]")
    print(
        sg.get_constraints_with_assignment_str(
            sym_map=preview_sym_map,
            include_vars=True,
            include_eval=True,
        )
    )


[Phase Order]
grid_0__execution_max_threads_pure_product ['sp_10_1', 'sp_9_1', 'sp_8_1', 'sp_7_1']
grid_0__execution_max_vthread_pure_product ['sp_8_0', 'sp_9_0', 'sp_10_0']
grid_0__execution_block_split_structure ['sp_7_3', 'sp_7_2', 'sp_7_0', 'sp_8_3', 'sp_8_2', 'sp_9_3', 'sp_9_2', 'sp_10_3', 'sp_10_2']
grid_0__memory_split_structure ['sp_11_1', 'sp_11_0', 'sp_12_1', 'sp_12_0']
grid_0__instruction_scaled_product_upper_bound ['sp_41_0']
grid_0__instruction_non_product_min ['sp_36_0']
grid_1__execution_non_product_direct_arm ['sp_26_0']
grid_1__execution_non_product_gate_vars ['<empty>']

=== grid_0__execution_max_threads_pure_product ===
[Phase Vars]
['sp_10_1', 'sp_9_1', 'sp_8_1', 'sp_7_1']

[Params At This Phase]
{'sp_10_1': 28, 'sp_9_1': 15, 'sp_8_1': 1, 'sp_7_1': 1}

[Fixed From Domains]
{'sp_7_0': 1, 'sp_7_2': 1, 'sp_7_3': 1, 'sp_9_0': 1, 'sp_9_2': 1, 'sp_9_3': 1}

[Domains After Phase]
{'sp_7_0': [1, 1], 'sp_7_1': 1, 'sp_7_2': [1, 1], 'sp_7_3': [1, 1], 'sp_8_0': [1, 2], 'sp_8_1'